# Data Exploration

More details here, but nothing like getting your hands dirty and exploring yourself! 
https://www.kaggle.com/code/ambrosm/esp-eda-which-makes-sense

In [2]:
import pandas as pd
import plotly.express as px

# If needed change the current working directory
# p = Path.cwd()
# print(p)
# import os
# from pathlib import Path
# os.chdir(p.parent)
# p = Path.cwd()
# print(p)

In [3]:
# Load datasets
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
data_dictionary = pd.read_csv('data/data_dictionary.csv')
example_submission = pd.read_csv('data/sample_submission.csv')

print('Training Shape:',train.shape)
print('Test Shape:',test.shape)
print('Data Dictionary Shape:',data_dictionary.shape)
print('Example Submission Shape:',example_submission.shape)

Training Shape: (28800, 60)
Test Shape: (3, 58)
Data Dictionary Shape: (59, 4)
Example Submission Shape: (3, 2)


### Dataset Description:
- The primary outcome of interest is event-free survival, represented by the variable **efs**, while the time to event-free survival is captured by the variable **efs_time**. These two variables together encode the target for a censored time-to-event analysis.
- The dataset consists of 59 variables related to hematopoietic stem cell transplantation (HSCT), encompassing a range of demographic and medical characteristics of both recipients and donors, such as age, sex, ethnicity, disease status, and treatment details.
- The data, which features equal representation across recipient racial categories including White, Asian, African-American, Native American, Pacific Islander, and More than One Race, was synthetically generated using the data generator from synthcity, trained on a large cohort of real CIBMTR data.

In [4]:
# ID refers to the identifier for each patient in the test dataset.
# prediction is the corresponding risk score generated by your model.
example_submission

,ID,prediction
0,28800,0.5
1,28801,0.5
2,28802,0.5


### What is a `Risk Score`? It looks like neither of the target variables given in the training data.

The prediction target consists of two parts:

- efs_time, always positive, is a time, measured in months.
- efs, always zero or one, indicates the presence or absence of an event:
    - efs=1 means "patient died exactly at time efs_time.
    - efs=0 means "patient still lives at time efs_time; in other words, "patient dies at an unknown time strictly greater than efs_time"
This situation is called "censored data": Samples of which we know the time of death are uncensored, and if we only know a lower bound for the time of death, the sample is (right-)censored.

Censoring is the main reason that this competition has a special metric and that we need special models. The competition is a regression task, but we know y_true for only half the samples. For the other (censored) half, all we know is lower bounds for y_true. One cannot compute a squared error based on y_true > 100 and y_pred == 120. RMSE and similar metrics cannot deal with that.

By the way, the column name is misleading: If a column is called "event-free survival", I'd expect that 0 means "patient died" and 1 means "patient lives", but that's wrong.

It's up to us to define a risk score. The scoring metric (concordance index) is rank-based. We don't really care about the actual values. 

Using only the efs column, we could see it as a survival probability (where 1 = certain death, 0 = certain survival).

But, since we also have efs_time, we should incorporate this feature too. e.g. efs = 0, but low efs_time (i.e. the study finished, or we lost contact with the patient), should have a higher score than efs=0 and a high efs_time.

Target variables:

In [5]:
target_variables = data_dictionary.iloc[-2:,:]
target_variables

,variable,description,type,values
57,efs,Event-free survival,Categorical,['Event' 'Censoring']
58,efs_time,"Time to event-free survival, months",Numerical,NaN


In [6]:
# EFS is either 0 or 1
train['efs'].value_counts()

efs
1.0    15532
0.0    13268
Name: count, dtype: int64

In [7]:
px.histogram(train, x='efs_time', title='EFS Time Distribution')

In [8]:
# Break out by EFS
px.histogram(train, x='efs_time', title='EFS Time Distribution, by EFS ',color='efs')

Patients with EFS = 1 have shorter EFS times than patients with EFS = 0.

In [survival analysis](https://en.wikipedia.org/wiki/Survival_analysis), an `Event` may be Death, disease occurrence, disease recurrence, recovery, or other experience of interest.

#### Confused - Think EFS = 1 means there was an event - i.e. the patient died during the study. 

Predictive Variables:

In [9]:
predictive_variables = data_dictionary.iloc[:-2,:]
print('Length:', len(predictive_variables))
print('Columns:')
display(predictive_variables)
print('Data Types:')
display(predictive_variables['type'].value_counts())

Length: 57
Columns:


,variable,description,type,values
0,dri_score,Refined disease risk index,Categorical,['Intermediate' 'High' 'N/A - non-malignant in...
1,psych_disturb,Psychiatric disturbance,Categorical,['Yes' 'No' nan 'Not done']
2,cyto_score,Cytogenetic score,Categorical,['Intermediate' 'Favorable' 'Poor' 'TBD' nan '...
3,diabetes,Diabetes,Categorical,['No' 'Yes' nan 'Not done']
4,hla_match_c_high,Recipient / 1st donor allele level (high resol...,Numerical,NaN
5,hla_high_res_8,Recipient / 1st donor allele-level (high resol...,Numerical,NaN
6,tbi_status,TBI,Categorical,"['No TBI' 'TBI + Cy +- Other' 'TBI +- Other, <..."
7,arrhythmia,Arrhythmia,Categorical,['No' nan 'Yes' 'Not done']
8,hla_low_res_6,Recipient / 1st donor antigen-level (low resol...,Numerical,NaN
9,graft_type,Graft type,Categorical,['Peripheral blood' 'Bone marrow']


Data Types:


type
Categorical    35
Numerical      22
Name: count, dtype: int64

#### Test Data - why are there only 3 patients?!

In [10]:
test  

,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,...,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10
0,28800,N/A - non-malignant indication,No,NaN,No,NaN,NaN,No TBI,No,6.0,...,90.0,No,NaN,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0
1,28801,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,...,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0
2,28802,N/A - non-malignant indication,No,NaN,No,2.0,8.0,No TBI,No,6.0,...,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0


In [11]:
## Example to incorporate efs and efs_time into a single predictive variable
## Take survival times, Rank them, and normalizes the ranks. It handles patients with and without events differently to ensure proper ranking and normalization.
## Source: https://www.kaggle.com/code/wjones3668/cibmtr-equity-in-post-hct-survival-predictions

train["y"] = train.efs_time.values
# Max survival time for patients with event (i.e. death) - think of this as the length of the study
mx = train.loc[train.efs==1,"efs_time"].max()
print('Max survival time for patient with event:',mx)
# Min survival time for patients without event (i.e. censored)
mn = train.loc[train.efs==0,"efs_time"].min()
print('Min survival time for patient without event:',mn)

train.loc[train.efs==0,"y"] = train.loc[train.efs==0,"y"] + mx - mn
train.y = train.y.rank()
train.loc[train.efs==0,"y"] += len(train)//2
train.y = train.y / train.y.max()


train
# px.histogram(train, color='efs')

Max survival time for patient with event: 120.009
Min survival time for patient without event: 3.212


,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,...,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,y
0,0,N/A - non-malignant indication,No,NaN,No,NaN,NaN,No TBI,No,6.0,...,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,42.356,0.875370
1,1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,...,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,4.672,0.101458
2,2,N/A - non-malignant indication,No,NaN,No,2.0,8.0,No TBI,No,6.0,...,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,19.793,0.726134
3,3,High,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,...,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,102.349,0.990463
4,4,High,No,NaN,No,2.0,8.0,No TBI,No,6.0,...,Related,MEL,8.0,No,2.0,No,10.0,0.0,16.223,0.711134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28795,28795,Intermediate - TED AML case <missing cytogenetics,NaN,Favorable,No,2.0,8.0,No TBI,No,6.0,...,NaN,"N/A, Mel not given",8.0,NaN,2.0,No,10.0,0.0,18.633,0.720567
28796,28796,High,No,Poor,Yes,1.0,4.0,No TBI,No,5.0,...,Related,"N/A, Mel not given",6.0,Yes,1.0,Yes,8.0,1.0,4.892,0.116285
28797,28797,TBD cytogenetics,NaN,Poor,NaN,2.0,8.0,No TBI,NaN,6.0,...,Unrelated,"N/A, Mel not given",8.0,NaN,2.0,No,10.0,0.0,23.157,0.749259
28798,28798,N/A - non-malignant indication,No,Poor,No,1.0,4.0,No TBI,No,3.0,...,Related,MEL,4.0,No,1.0,No,5.0,0.0,52.351,0.916944
